# TP — Morphologie mathématique binaire

Ce notebook traite les consignes visibles du sujet : **érosion, dilatation, ouverture et fermeture**, puis comparaison de plusieurs formes et tailles d'éléments structurants. L'image originale du sujet n'étant pas disponible, les expériences utilisent une image binaire synthétique reproductible ; il faudra remplacer cette image par celle du cours pour reproduire exactement les résultats attendus.

On considère les pixels de premier plan comme `True`. L'élément structurant est centré sur son pixel central et le fond hors image vaut `False`.

## 1. Définitions

Pour un ensemble de pixels `A` et un élément structurant `B` :

- **Érosion** : un pixel reste au premier plan si tous les pixels couverts par `B` sont dans `A`. Elle rétrécit les objets et élimine les détails plus petits que `B`.
- **Dilatation** : un pixel passe au premier plan si au moins un pixel couvert par `B` appartient à `A`. Elle agrandit les objets et peut relier des composantes proches.
- **Ouverture** : érosion suivie d'une dilatation avec le même `B`. Elle supprime les petits objets/bruits tout en restaurant en partie la taille des objets conservés.
- **Fermeture** : dilatation suivie d'une érosion avec le même `B`. Elle comble les petits trous et relie les petites coupures.

Les éléments utilisés sont symétriques, ce qui rend leur orientation sans ambiguïté.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy {np.__version__}")

In [ ]:
def morphologique(image, element, operation):
    """Applique une érosion ou dilatation binaire avec fond extérieur nul."""
    image = np.asarray(image, dtype=bool)
    element = np.asarray(element, dtype=bool)
    if image.ndim != 2 or element.ndim != 2:
        raise ValueError("L'image et l'élément structurant doivent être 2D.")
    if any(size % 2 == 0 for size in element.shape):
        raise ValueError("Les dimensions de l'élément structurant doivent être impaires.")
    if not element[element.shape[0] // 2, element.shape[1] // 2]:
        raise ValueError("Le pixel central de l'élément structurant doit être actif.")
    if operation not in {"erosion", "dilatation"}:
        raise ValueError("operation doit être 'erosion' ou 'dilatation'.")

    pad_y, pad_x = element.shape[0] // 2, element.shape[1] // 2
    padded = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), constant_values=False)
    result = np.ones_like(image) if operation == "erosion" else np.zeros_like(image)
    for y, x in np.argwhere(element):
        voisinage = padded[y:y + image.shape[0], x:x + image.shape[1]]
        if operation == "erosion":
            result &= voisinage
        else:
            result |= voisinage
    return result


def erosion(image, element):
    return morphologique(image, element, "erosion")


def dilatation(image, element):
    return morphologique(image, element, "dilatation")


def ouverture(image, element):
    image = np.asarray(image, dtype=bool)
    pad_y, pad_x = element.shape[0] // 2, element.shape[1] // 2
    padded = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), constant_values=False)
    resultat = dilatation(erosion(padded, element), element)
    return resultat[pad_y:pad_y + image.shape[0], pad_x:pad_x + image.shape[1]]


def fermeture(image, element):
    image = np.asarray(image, dtype=bool)
    pad_y, pad_x = element.shape[0] // 2, element.shape[1] // 2
    padded = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), constant_values=False)
    resultat = erosion(dilatation(padded, element), element)
    return resultat[pad_y:pad_y + image.shape[0], pad_x:pad_x + image.shape[1]]


def element_structurant(forme, taille):
    """Construit un élément structurant centré de taille impaire."""
    if taille < 1 or taille % 2 == 0:
        raise ValueError("La taille doit être un entier positif impair.")
    if forme == "carre":
        return np.ones((taille, taille), dtype=bool)
    if forme == "croix":
        elem = np.zeros((taille, taille), dtype=bool)
        elem[taille // 2, :] = True
        elem[:, taille // 2] = True
        return elem
    if forme == "disque":
        coord = np.arange(taille) - taille // 2
        y, x = np.meshgrid(coord, coord, indexing="ij")
        return x * x + y * y <= (taille // 2) ** 2
    if forme == "ligne_horizontale":
        return np.ones((1, taille), dtype=bool)
    raise ValueError(f"Forme inconnue : {forme}")

## 2. Image binaire d'expérience

L'image jouet contient des rectangles, un disque, une ligne fine, un petit trou et du bruit isolé. La graine fixe garantit des résultats identiques à chaque exécution.

In [ ]:
n = 128
yy, xx = np.indices((n, n))
image = np.zeros((n, n), dtype=bool)
image[18:48, 15:48] = True                 # rectangle
image[(yy - 39) ** 2 + (xx - 88) ** 2 <= 16 ** 2] = True  # disque
image[77:108, 18:23] = True                # rectangle étroit
image[91:94, 62:111] = True                # ligne fine
image[28:34, 26:32] = False                # petit trou dans le rectangle
rng = np.random.default_rng(2026)
image[rng.integers(0, n, 35), rng.integers(0, n, 35)] = True  # bruit ponctuel

plt.figure(figsize=(5, 5))
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("Image binaire synthétique")
plt.axis("off");
print(f"Pixels de premier plan : {image.sum()} / {image.size}")

## 3. Les quatre opérations

Comparaison sur un disque de taille 7. L'érosion enlève les détails fins, la dilatation élargit les régions, l'ouverture retire notamment les pixels isolés et la fermeture réduit le trou intérieur.

In [ ]:
element = element_structurant("disque", 7)
resultats = [
    ("Image originale", image),
    ("Érosion", erosion(image, element)),
    ("Dilatation", dilatation(image, element)),
    ("Ouverture", ouverture(image, element)),
    ("Fermeture", fermeture(image, element)),
]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, (titre, resultat) in zip(axes, resultats):
    ax.imshow(resultat, cmap="gray", vmin=0, vmax=1)
    ax.set_title(titre)
    ax.axis("off")
plt.tight_layout();

## 4. Influence de la forme et de la taille

On compare l'ouverture et la fermeture pour quatre formes et des tailles 3, 5 et 7. Une croix préserve mieux les structures alignées horizontalement/verticalement ; une ligne horizontale agit principalement sur les variations horizontales ; carré et disque produisent des effets plus isotropes, avec des contours différents.

In [ ]:
formes = ["carre", "croix", "disque", "ligne_horizontale"]
tailles = [3, 5, 7]
fig, axes = plt.subplots(len(formes), len(tailles) + 1, figsize=(12, 12))
for row, forme in enumerate(formes):
    axes[row, 0].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"{forme}\noriginale")
    axes[row, 0].axis("off")
    for col, taille in enumerate(tailles, start=1):
        elem = element_structurant(forme, taille)
        resultat = ouverture(image, elem)
        axes[row, col].imshow(resultat, cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(f"ouverture, taille {taille}")
        axes[row, col].axis("off")
plt.suptitle("Ouverture : influence de la forme et de la taille", y=1.01)
plt.tight_layout();

fig, axes = plt.subplots(len(formes), len(tailles) + 1, figsize=(12, 12))
for row, forme in enumerate(formes):
    axes[row, 0].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"{forme}\noriginale")
    axes[row, 0].axis("off")
    for col, taille in enumerate(tailles, start=1):
        elem = element_structurant(forme, taille)
        resultat = fermeture(image, elem)
        axes[row, col].imshow(resultat, cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(f"fermeture, taille {taille}")
        axes[row, col].axis("off")
plt.suptitle("Fermeture : influence de la forme et de la taille", y=1.01)
plt.tight_layout();

## 5. Mesure de la quantité de premier plan

Le nombre de pixels permet de quantifier les changements, mais ne décrit pas à lui seul la géométrie : deux images peuvent avoir le même nombre de pixels et des composantes différentes.

In [ ]:
print(f"{'Forme':<20} {'Taille':>6} {'Érosion':>10} {'Dilatation':>12} {'Ouverture':>11} {'Fermeture':>11}")
for forme in formes:
    for taille in tailles:
        elem = element_structurant(forme, taille)
        mesures = [
            erosion(image, elem).sum(),
            dilatation(image, elem).sum(),
            ouverture(image, elem).sum(),
            fermeture(image, elem).sum(),
        ]
        print(f"{forme:<20} {taille:>6} {mesures[0]:>10} {mesures[1]:>12} {mesures[2]:>11} {mesures[3]:>11}")

## 6. Vérifications et conclusion

Pour ces éléments structurants centrés et symétriques, l'érosion et l'ouverture sont anti-extensives (leur résultat est inclus dans l'image), tandis que la dilatation et la fermeture sont extensives. L'ouverture et la fermeture sont aussi idempotentes : les répéter avec le même élément ne change plus le résultat. La taille doit être choisie selon l'échelle du bruit ou des détails à traiter : un élément trop grand détruit des structures utiles ou fusionne des objets.

In [ ]:
for forme in formes:
    for taille in tailles:
        elem = element_structurant(forme, taille)
        er = erosion(image, elem)
        di = dilatation(image, elem)
        op = ouverture(image, elem)
        fe = fermeture(image, elem)
        assert np.all(er <= image), (forme, taille, "érosion anti-extensive")
        assert np.all(image <= di), (forme, taille, "dilatation extensive")
        assert np.all(op <= image), (forme, taille, "ouverture anti-extensive")
        assert np.all(image <= fe), (forme, taille, "fermeture extensive")
        assert np.array_equal(ouverture(op, elem), op), (forme, taille, "ouverture idempotente")
        assert np.array_equal(fermeture(fe, elem), fe), (forme, taille, "fermeture idempotente")

print(f"Vérifications réussies : {len(formes) * len(tailles)} couples forme/taille, propriétés morphologiques satisfaites.")